# Local GPU student evaluation: Qwen3-0.6B

This notebook runs the same aligned GPQA/SCUA conditions as the API experiment, using a model loaded directly on the GPU server. Every run is saved under `gpu_experiments/outputs/`.

In [ ]:
from pathlib import Path
import json
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'content conditions').is_dir() and (candidate / 'gpu_experiments').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'gpu_experiments'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from gpu_experiments import ExperimentConfig, ModelConfig, load_model, preview_task, run_experiment
from student_eval import CONDITION_FILES
print('Project root:', REPO_ROOT)

## Experiment variables

`NUM_ROWS` is the number of aligned questions per condition. Use `None` for all 448 rows. Start with one row and a batch size of one, then increase the batch size to fit the server GPU.

In [ ]:
MODEL_ID = 'Qwen/Qwen3-0.6B'
TARGET_CONDITIONS = [0]       # Any subset of 0..6
NUM_ROWS = 1                  # Rows per condition; None means all rows
START_ROW = 0
BATCH_SIZE = 1                # Increase after the smoke test
MAX_NEW_TOKENS = 1024
TEMPERATURE = 0.0
ENABLE_THINKING = False       # Short JSON mode for the baseline

MODEL_CONFIG = ModelConfig(
    model_id=MODEL_ID,
    dtype='bfloat16',          # Change to float16 if BF16 is unsupported
    device_map='auto',
    attention_implementation=None,  # Or 'flash_attention_2' if installed
    cache_dir=None,
    local_files_only=False,
)
CONFIG = ExperimentConfig(
    model=MODEL_CONFIG,
    condition_ids=tuple(TARGET_CONDITIONS),
    num_rows=NUM_ROWS,
    start_row=START_ROW,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    enable_thinking=ENABLE_THINKING,
)
for condition_id in TARGET_CONDITIONS:
    print(f'{condition_id}: {CONDITION_FILES[condition_id]}')

## Preview one prompt (does not load the model)

In [ ]:
preview = preview_task(REPO_ROOT, CONFIG)
print('Request key:', preview['request_key'])
print('Reference answer (not sent):', preview['reference_answer'])
print('\n--- Prompt sent to the model ---\n')
print(preview['prompt'])

## Load once

This is the first cell that downloads model files, initializes CUDA, and allocates GPU memory.

In [ ]:
loaded_model = load_model(MODEL_CONFIG)
print('Loaded:', loaded_model.config.model_id)

## Run and save

In [ ]:
run_dir, results, summary = run_experiment(
    REPO_ROOT, EXPERIMENT_DIR, CONFIG, loaded_model=loaded_model
)
print('Saved run:', run_dir)
print(json.dumps(summary, indent=2))

## Result preview

In [ ]:
for result in results[:5]:
    print(json.dumps({
        'request_key': result['request_key'],
        'prediction': result.get('prediction'),
        'reference_answer': result['answer_key'],
        'is_correct': result.get('is_correct'),
        'reason': result.get('reason'),
        'usage': result.get('usage'),
        'latency_seconds': result.get('latency_seconds'),
        'error': result.get('error'),
    }, ensure_ascii=False, indent=2))